# 00 — Collect "Unknown" Images

Before training I needed a collection of images that clearly do **not** show a cédula or a passport
— this is the `unknown` class (label 3) that teaches the model to say *"I don't recognise this"*.

In addition of manually downloading photos I used [Lorem Picsum](https://picsum.photos), which serves
royalty-free random photographs over a simple HTTP endpoint.
The images land in `data/raw/unknown/` and are treated as negative examples throughout training.

> **Run this once only.**  Re-running downloads a different batch of images
> (Picsum randomises the output), which would silently change the dataset and break reproducibility.
> The `cache=i+103` offset in the loop is a seed trick that gives deterministic URLs if you
> need to re-fetch exactly the same images.

In [2]:
import requests, os, sys
from pathlib import Path

In [3]:
# Add the project-level global/ folder to sys.path so that
# `import config` works without installing anything as a package.
GLOBAL_PATH = str(Path.cwd().parent.parent / "global")
if GLOBAL_PATH not in sys.path:
    sys.path.append(GLOBAL_PATH)

%reload_ext autoreload
%autoreload 2
import config

In [4]:
raw_unknown_dir = Path(config.RAW_DIR) / "unknown"
print(f"Downloading images to: {raw_unknown_dir.absolute()}")

In [7]:
# picsum.photos returns a random stock photo at the given resolution.
# The cache= parameter acts as a seed — the same number always gives
# the same image, so the dataset is reproducible.
for i in range(37):
    url = f"https://picsum.photos/512?cache={i+103}"

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            file_path = raw_unknown_dir / f"unknown_{i+103}.jpg"
            with open(file_path, "wb") as f:
                f.write(response.content)
            if (i + 1) % 10 == 0:
                print(f"Downloaded {i + 1}/37 images...")
        else:
            print(f"Failed at image {i}: HTTP {response.status_code}")
    except Exception as e:
        print(f"Error at image {i}: {e}")

Error at image 8: HTTPSConnectionPool(host='picsum.photos', port=443): Read timed out. (read timeout=10)
Downloaded 10/60 images...
Error at image 18: HTTPSConnectionPool(host='picsum.photos', port=443): Read timed out. (read timeout=10)
Downloaded 20/60 images...
Downloaded 30/60 images...
Error at image 30: HTTPSConnectionPool(host='picsum.photos', port=443): Read timed out. (read timeout=10)
